In [1]:
!git clone https://github.com/OpenMOSS/MOSS-TTS.git
%cd MOSS-TTS
!pip install datasets soundfile matplotlib numpy -q wandb
!apt-get install -y ffmpeg -q

fatal: destination path 'MOSS-TTS' already exists and is not an empty directory.
/content/MOSS-TTS
Reading package lists...
Building dependency tree...
Reading state information...
ffmpeg is already the newest version (7:4.4.2-0ubuntu0.22.04.1).
0 upgraded, 0 newly installed, 0 to remove and 42 not upgraded.


In [2]:
MODEL_NAME  = "OpenMOSS-Team/MOSS-TTS" # Delay 8B
# MODEL_NAME = "OpenMOSS-Team/MOSS-TTS-Local-Transformer" # Local 1.7B

BATCH_SIZES = [1, 4, 8, 16, 64, 128, 256] # tested batch sizes
REPEATS = 1 #num times you want to redo batch size exp
MAX_NEW_TOKENS = 200 
FIXED_TEXT = "The weather is so nice today and the birds are singing in the trees."

In [3]:
import importlib.util
import wandb

import json
import random
import time
from pathlib import Path
import numpy as np
import soundfile as sf
import torch
from transformers import AutoModel, AutoProcessor
import time
import torch
import numpy as np
import json
from pathlib import Path

# required SDPA backend flags from official MOSS-TTS docs
torch.backends.cuda.enable_cudnn_sdp(False)
torch.backends.cuda.enable_flash_sdp(True)
torch.backends.cuda.enable_mem_efficient_sdp(True)
torch.backends.cuda.enable_math_sdp(True)

device = 'cuda' if torch.cuda.is_available() else 'cpu'

In [4]:
# run text through mode and get metrics for latency and audio duration
def run_batch(model, processor, texts, max_new_tokens=200):
    audioTotal = 0.0

    torch.cuda.synchronize()

    t0 = time.perf_counter()

    batchConvo = [[processor.build_user_message(text= t)] for t in texts]
    batch = processor(batchConvo, mode='generation')
    
    input_ids = batch['input_ids'].to(device)
    attention_mask = batch['attention_mask'].to(device)

    with torch.no_grad():
        outputs = model.generate(input_ids= input_ids, attention_mask= attention_mask, max_new_tokens= max_new_tokens)

    # decode audio and calc total audio duration in sec
    messages = processor.decode(outputs)
    sampleRate = processor.model_config.sampling_rate

    for msg in messages:
        audio = msg.audio_codes_list[0]
        audioTotal += audio.shape[-1] / sampleRate

    torch.cuda.synchronize()

    return time.perf_counter() - t0, audioTotal

In [5]:
#load model and processor
processor = AutoProcessor.from_pretrained(MODEL_NAME, trust_remote_code= True)

if hasattr(processor, 'audio_tokenizer'):
    processor.audio_tokenizer = processor.audio_tokenizer.to(device).eval()

model = AutoModel.from_pretrained(MODEL_NAME, trust_remote_code= True, torch_dtype= torch.bfloat16).to(device).eval()

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:103: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
You are not authenticated with the Hugging Face Hub in this notebook.
If the error persists, please let us know by opening an issue on GitHub (https://github.com/huggingface/huggingface_hub/issues/new).
  warnings.warn(


processor_config.json:   0%|          | 0.00/145 [00:00<?, ?B/s]

chat_template.jinja:   0%|          | 0.00/352 [00:00<?, ?B/s]

processing_moss_tts.py: 0.00B [00:00, ?B/s]

configuration_moss_tts.py: 0.00B [00:00, ?B/s]

...

inference_utils.py: 0.00B [00:00, ?B/s]

A new version of the following files was downloaded from https://huggingface.co/OpenMOSS-Team/MOSS-TTS:
- inference_utils.py
. Make sure to double-check they do not contain any added malicious code. To avoid downloading new versions of the code file, you can pin a revision.
A new version of the following files was downloaded from https://huggingface.co/OpenMOSS-Team/MOSS-TTS:
- modeling_moss_tts.py
- inference_utils.py
. Make sure to double-check they do not contain any added malicious code. To avoid downloading new versions of the code file, you can pin a revision.


model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 4 files:   0%|          | 0/4 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/463 [00:00<?, ?it/s]

In [6]:
# get fixed text batches for max batch size and repeats to avoid data loading variance
textAll = [FIXED_TEXT] * (max(BATCH_SIZES) * REPEATS)



print(f'{"BS":>4}  {"wall(s)":>10}  {"audio(s)":>10}  {"tput(a/s)":>10}  {"speedup":>8}')
print(f'{"-"*4}  {"-"*10}  {"-"*10}  {"-"*10}  {"-"*8}')

wandbRun = wandb.init(project="hpml-final-project", name= f"hf-{MODEL_NAME}-throughput", config={"model": MODEL_NAME, 
                                                                                                 "fixed_text": FIXED_TEXT,  
                                                                                                 "max_ar_tokens": MAX_NEW_TOKENS,
                                                                                                 "batch_sizes": BATCH_SIZES,
                                                                                                 "n_repeats": REPEATS})

# run batches and collect results for each batch size, then save to json
tp0 = None

for bs in BATCH_SIZES:
    walls, audios = [], []

    for i in range(REPEATS):

        text = textAll[i * bs:(i + 1) * bs]

        wall, audio = run_batch(model, processor, text, MAX_NEW_TOKENS)
        walls.append(wall)
        audios.append(audio)

    meanWall = float(np.mean(walls))
    meanAudio = float(np.mean(audios))

    # throughput = audio / wall time
    tp = meanAudio / meanWall if meanWall > 0 else 0

    #first batch
    if tp0 is None:
        tp0 = tp

    speedup = tp / tp0 if tp0 > 0 else 1

    wandbRun.log({"bs": bs, "wall_s": meanWall, "audio_s": meanAudio, "throughput_audio_per_s": tp, "speedup": speedup})
    
    
    print(f'{bs:>4}  {meanWall:>10.2f}  {meanAudio:>10.2f}  {tp:>10.3f}    {speedup:>7.2f}x')

wandbRun.finish()

  BS     wall(s)    audio(s)   tput(a/s)   speedup
----  ----------  ----------  ----------  --------


wandb: [wandb.login()] Loaded credentials for https://api.wandb.ai from /root/.netrc.
wandb: Currently logged in as: ac5905 (ac5905-columbia-university) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


Generating bs1 ...:  41%|████      | 82/200 [00:05<00:08, 14.73it/s]


   1        5.93        3.92       0.661       1.00x


Generating bs4 ...:  52%|█████▏    | 103/200 [00:06<00:06, 15.49it/s]


...

Generating bs128 ...: 100%|██████████| 200/200 [00:16<00:00, 11.94it/s]


 128       30.87      648.64      21.014      31.81x


Generating bs256 ...: 100%|██████████| 200/200 [00:24<00:00,  8.06it/s]


 256       53.18     1304.88      24.538      37.14x


audio_s,▁▁▁▁▃▄█
bs,▁▁▁▁▃▄█
speedup,▁▂▂▃▇▇█
throughput_audio_per_s,▁▂▂▃▇▇█
wall_s,▁▁▁▂▂▅█
audio_s,1304.88
bs,256
speedup,37.14018
throughput_audio_per_s,24.53775
wall_s,53.17846
